# UNO Vision

Image Analysis and Pattern Recognition - Final Project

Group 34

Fiona Dematraz & Killian Ballifard & Gabriel Marival

## 1. Introduction

Automatically understanding the state of a game from a visual input is a complex problem that combines object detection, classification, and spatial reasoning. For the project, this problem is placed in the context of the game UNO, with the goal of inferring the complete state of a game from a single top-down photograph of the table.

From a single image, the system must predict the central card in play, the active player, and the set of cards held by each of the four players. Players occupy fixed spatial positions around the table, and the active player is indicated by a physical token placed next to their hand. The dataset presents two main difficulties: the background variability (plain white vs. noisy patterned) and the card layout complexity (non-overlapping vs. overlapping).

To address this, a pipeline was designed with two parallel branches: one for detecting the active token via color segmentation and shape filtering, and the other for card recognition, which divides the image into spatial sectors, converts them to a 4-color HSV representation, and classifies each sector using a CNN with attention pooling. The final game state is assembled by combining the two outputs and evaluated using a weighted score of center card accuracy (10%), active player accuracy (10%), and per-player F1 score (80%).

## 2. Dataset

The dataset consists of top-down photographs taken during actual UNO games, captured under a variety of conditions. There are two types of backgrounds: plain white backgrounds, which provide a clean, easy-to-segment environment, and backgrounds with busy patterns (such as a tablecloth with leaf patterns), which make isolating the cards significantly more difficult. For each background type, the dataset includes both non-overlapping and overlapping card layouts, with the latter being more challenging to process because the card edges are partially obscured.

Each image depicts up to four players positioned at fixed locations around the table (bottom, right, top, left), though not all images necessarily contain four players. In the center of the table is the active discard pile. The active player is indicated by a physical token—a small dark cube or a yellow coin—placed next to their hand of cards.

The cards fall into 54 distinct categories: number cards (0–9) in four colors (red, green, blue, yellow), as well as action cards (Pass, Reverse, Draw Two) and wild cards (Wild, Wild Draw Four).

## 3. Methodology 

### 3.1 Overview

### 3.2 Active Token Detection


The goal of this branch is to find the active player token in the image and assign it to one of the four player zones.

#### Step 1 : Background Detection

The first step determines which type of token to look for. The four corners of the image are sampled, which always contain background and never cards. If the mean brightness of these corners exceeds 200, the scene is classified as a white background; otherwise, it is classified as a flower background. This distinction lets know which token types are: white background always use a black rectangular token, while flower background always use a yellow circular token.

#### Step 2 : Color Masking

Depending on the background type, a binary mask is computed on the full image.

- **White background** : an HSV threshold with `V < 130` and `S < 90` is applied to isolate dark pixels corresponding to the black token. Morphological closing (15×15 kernel) fills internal gaps, followed by opening (7×7 kernel) to remove noise.
- **Flower background** : the preprocessing pipeline's color segmentation function, already calibrated for UNO card colors,is used to produce a soft yellow confidence map, which is then thresholded at 40 to obtain a binary mask.

#### Step 3 : Blob Selection

For each contour extracted from the binary mask, a set of shape descriptors is computed:

| Descriptor | Formula | Ideal value |
|---|---|---|
| Circularity | `4π × area / perimeter²` | 1.0 = perfect circle |
| Solidity | `area / convex_hull_area` | 1.0 = fully solid |
| Aspect ratio | `max(w,h) / min(w,h)` | 1.0 = square |
| Rect fill *(black only)* | `area / minAreaRect_area` | 1.0 = solid rectangle |
| Yellow density *(yellow only)* | fraction of yellow pixels inside blob | filters hollow symbols |

**Yellow token selection** : a candidate must satisfy:
- `circularity > 0.65` (round shape)
- `solidity > 0.65` (solid interior)
- `yellow_density > 0.55` (uniformly filled — rejects Skip card circles which have white holes)

The blob maximizing `circularity × solidity × yellow_density` is selected.

**Black token selection** : a candidate must satisfy:
- `circularity < 0.82` (non-circular, i.e. rectangular)
- `aspect_ratio > 1.1` (elongated)
- `rect_fill > 0.65` (solid rectangle, not a hollow card symbol)

Among valid candidates, the largest blob by area is retained, as the token is the biggest dark solid object in the scene.

#### Step 4 : Player Assignment

The centroid `(cx, cy)` of the detected token is mapped to the nearest player zone using Euclidean distance:

```
active_player = argmin_p  distance(token_center, player_zone_center_p)
```

The four zone centers are fixed image coordinates corresponding to P1 (bottom), P2 (right), P3 (top), and P4 (left).



### 3.3 Pre-processing 

### 3.4 CNN

## 4. Results 

## 5. Discussion

## 7. Conclusion